# EO4040 — Tarea 1: Calidad y limpieza de datos (MASTER)
## Dataset: ENIGH 2022 — concentradohogar

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('✅ Librerías cargadas')

---
## Pregunta 1 — Cargar el archivo (10 pts)

In [ ]:
# SOLUCIÓN
df = pd.read_csv('concentradohogar.csv', encoding='latin-1')
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

---
## Pregunta 2 — Tabla de calidad (10 pts)

In [ ]:
# SOLUCIÓN
tabla_calidad = pd.DataFrame({
    'variable':      df.columns.tolist(),
    'tipo_dato':     [str(df[c].dtype) for c in df.columns],
    'pct_faltantes': [(df[c].isnull().sum() / len(df) * 100).round(2) for c in df.columns],
    'n_unicos':      [df[c].nunique() for c in df.columns],
})
tabla_calidad.head(10)

---
## Pregunta 3 — Identificar problemas (10 pts)

In [ ]:
# SOLUCIÓN
vars_con_faltantes = tabla_calidad[tabla_calidad['pct_faltantes'] > 0]['variable'].tolist()
n_duplicados       = int(df.duplicated(subset='folioviv').sum())
col_mas_faltantes  = tabla_calidad.loc[tabla_calidad['pct_faltantes'].idxmax(), 'variable']
print(f'Variables con faltantes: {len(vars_con_faltantes)}')
print(f'Duplicados en folioviv:  {n_duplicados}')
print(f'Columna con más faltantes: {col_mas_faltantes}')

---
## Pregunta 4 — Crear copia y eliminar duplicados (10 pts)

In [ ]:
# SOLUCIÓN
df_clean = df.copy()
df_clean = df_clean.drop_duplicates(subset='folioviv', keep='first')
print(f'Filas originales:                  {len(df):,}')
print(f'Filas después de quitar duplicados: {len(df_clean):,}')

---
## Pregunta 5 — Renombrar e imputar ingreso corriente (10 pts)

In [ ]:
# SOLUCIÓN
df_clean = df_clean.rename(columns={'ing_cor': 'ingcor'})
df_clean['ingcor_faltante'] = df_clean['ingcor'].isnull().astype(int)
mediana_ingcor = df_clean['ingcor'].median()
df_clean['ingcor'] = df_clean['ingcor'].fillna(mediana_ingcor)
print(f'Valores faltantes en ingcor después de imputar: {df_clean["ingcor"].isnull().sum()}')
print(f'Hogares con ingreso imputado (ingcor_faltante=1): {df_clean["ingcor_faltante"].sum()}')

---
## Pregunta 6 — Corregir tipo de est_socio (10 pts)

In [ ]:
# SOLUCIÓN
cat_type = pd.CategoricalDtype(categories=[1, 2, 3, 4], ordered=True)
df_clean['est_socio'] = df_clean['est_socio'].astype(cat_type)
print(f'Tipo de est_socio: {df_clean["est_socio"].dtype}')
print(f'¿Ordenada?: {df_clean["est_socio"].cat.ordered}')

---
## Pregunta 7 — Reporte comparativo (10 pts)

In [ ]:
# SOLUCIÓN
reporte = pd.DataFrame({
    'dataset':   ['original', 'limpio'],
    'n_filas':   [len(df), len(df_clean)],
    'total_nan': [df.isnull().sum().sum(), df_clean.isnull().sum().sum()],
})
reporte

---
## Pregunta 8 — Guardar el dataset limpio (10 pts)

In [ ]:
# SOLUCIÓN
df_clean.to_csv('enigh2022_limpio.csv', index=False)
df_verificacion = pd.read_csv('enigh2022_limpio.csv')
print(f'Archivo guardado: {df_verificacion.shape[0]:,} filas × {df_verificacion.shape[1]} columnas')